# Lab 21 — BONUS ALL (B1–B5)

Chọn GPU runtime, thêm `HF_TOKEN` và `GITHUB_TOKEN` trong Colab Secrets, rồi chạy tuần tự. Các stage train có resume.

In [ ]:
# @title Setup
import os, pathlib, subprocess, sys, torch
REPO = "https://github.com/ashura102938475/Day21-Track3-Finetuning-Lab.git"
WORK = pathlib.Path("/content/Day21-Track3-Finetuning-Lab")
if not WORK.exists(): subprocess.run(["git", "clone", REPO, str(WORK)], check=True)
os.chdir(WORK)
subprocess.run(["git", "pull", "--ff-only"], check=True)
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r", "requirements.txt"], check=True)
if not torch.cuda.is_available(): raise RuntimeError("Chọn Runtime > Change runtime type > GPU")
# Match the already-frozen core model; Colab GPU is hardware, not a reason to change the experiment.
os.environ["COMPUTE_TIER"] = "LAPTOP"
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"
print("GPU ready:", torch.cuda.get_device_name(0))

In [ ]:
# @title Secrets
import os
from google.colab import userdata
HF_TOKEN = userdata.get("HF_TOKEN")
GITHUB_TOKEN = userdata.get("GITHUB_TOKEN")
if not HF_TOKEN or not GITHUB_TOKEN: raise RuntimeError("Add HF_TOKEN and GITHUB_TOKEN to Colab Secrets")
os.environ["HF_TOKEN"] = HF_TOKEN
os.environ["GITHUB_TOKEN"] = GITHUB_TOKEN
print("Secrets loaded (values hidden)")

In [ ]:
# @title B2 — custom + reasoning datasets
import subprocess, sys
subprocess.run([sys.executable, "scripts/bonus_data.py", "--write"], check=True)
subprocess.run([sys.executable, "scripts/bonus_verify.py", "--allow-unrun-gpu"], check=True)

In [ ]:
# @title B1 — merge + hot-swap
import subprocess, sys
subprocess.run([sys.executable, "-u", "scripts/bonus_stage.py", "b1"], check=True)

In [ ]:
# @title B3 — reasoning-trace collapse (resumable)
import subprocess, sys
subprocess.run([sys.executable, "-u", "scripts/bonus_stage.py", "b3"], check=True)

In [ ]:
# @title B4 — controlled rank sweep (resumable)
import subprocess, sys
subprocess.run([sys.executable, "-u", "scripts/bonus_stage.py", "b4"], check=True)

In [ ]:
# @title Verify first, then B5 publish
import os, subprocess, sys
subprocess.run([sys.executable, "scripts/verify.py"], check=True)
subprocess.run([sys.executable, "scripts/bonus_verify.py", "--allow-unrun-gpu"], check=True)
subprocess.run([sys.executable, "scripts/publish_bonus.py"], check=True, env=os.environ.copy())
subprocess.run([sys.executable, "scripts/bonus_verify.py"], check=True)
paths = ["data/CUSTOM_DATASET.md", "data/train_custom_ai_course.jsonl", "data/train_reasoning_trace.jsonl", "data/trace_holdout.jsonl", "results/merge_check.json", "results/bonus", "submission/REPORT.md", "LINKS.md"]
subprocess.run(["git", "add", "-f", "--", *paths], check=True)
commit_env = {**os.environ, "GIT_AUTHOR_NAME": "NGUYỄN ANH TRÀ", "GIT_AUTHOR_EMAIL": "tra01020407@gmail.com", "GIT_COMMITTER_NAME": "NGUYỄN ANH TRÀ", "GIT_COMMITTER_EMAIL": "tra01020407@gmail.com"}
if subprocess.run(["git", "diff", "--cached", "--quiet"]).returncode:
    subprocess.run(["git", "commit", "-m", "Complete Lab 21 bonuses B1-B5"], check=True, env=commit_env)
    if subprocess.run(["git", "remote", "get-url", "mine"], capture_output=True).returncode:
        subprocess.run(["git", "remote", "add", "mine", REPO], check=True)
    else:
        subprocess.run(["git", "remote", "set-url", "mine", REPO], check=True)
    helper = '!f() { echo username=x-access-token; echo password=$GITHUB_TOKEN; }; f'
    subprocess.run(["git", "-c", "credential.helper=" + helper, "push", "mine", "HEAD:main"], check=True, env=os.environ.copy())
print("All bonus evidence published")